# Feature Engineering — Construcción de Variables Predictivas

Construimos las features que alimentarán el modelo predictivo.

**Regla fundamental:** para predecir el partido del día X, 
solo usamos información disponible *antes* del día X.
Violar esto produce data leakage — métricas infladas que 
no se sostienen en producción.

**Features a construir:**
- Forma reciente de cada equipo (últimos 5 partidos)
- Promedio de goles anotados y recibidos (últimos 5 partidos)
- Historial head-to-head entre los dos equipos
- Tipo de partido (amistoso vs oficial, cancha neutral)

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/results_clean.csv', parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f"✅ {len(df):,} partidos cargados")
print(f"   Rango: {df['date'].min().date()} → {df['date'].max().date()}")

✅ 49,477 partidos cargados
   Rango: 1872-11-30 → 2026-06-27


## 1. Features simples

Las más directas — ya están casi en el dataset, solo hay que 
transformarlas al formato que espera el modelo (numérico).

In [2]:
# Trabajamos sobre una copia para no tocar el dataset limpio
df_features = df.copy()

# is_neutral: el dataset ya tiene columna bool, la convertimos a int
# Los modelos de sklearn esperan números, no True/False
df_features['is_neutral'] = df_features['neutral'].astype(int)

# is_friendly: 1 si es amistoso, 0 si es partido oficial
df_features['is_friendly'] = (df_features['tournament'] == 'Friendly').astype(int)

print("Features simples creadas:")
print(df_features[['date', 'home_team', 'away_team', 
                    'is_neutral', 'is_friendly']].head(8))
print(f"\nPartidos neutrales:  {df_features['is_neutral'].sum():,}")
print(f"Partidos amistosos:  {df_features['is_friendly'].sum():,}")

Features simples creadas:
        date home_team away_team  is_neutral  is_friendly
0 1872-11-30  Scotland   England           0            1
1 1873-03-08   England  Scotland           0            1
2 1874-03-07  Scotland   England           0            1
3 1875-03-06   England  Scotland           0            1
4 1876-03-04  Scotland   England           0            1
5 1876-03-25  Scotland     Wales           0            1
6 1877-03-03   England  Scotland           0            1
7 1877-03-05     Wales  Scotland           0            1

Partidos neutrales:  13,121
Partidos amistosos:  18,388


## 2. Historial por equipo (team_history)

Para calcular la forma reciente necesitamos ver el historial 
de cada equipo en orden cronológico. El problema: en el dataset 
original cada partido tiene local y visitante en la misma fila.

Solución: duplicamos cada partido — una fila desde la perspectiva 
del local y otra desde la del visitante. Así podemos ordenar 
por equipo + fecha y mirar hacia atrás limpiamente.

In [3]:
# Construimos la vista unificada de historial por equipo
# Cada partido genera DOS filas: una por equipo

# Perspectiva del equipo local
home_view = df[['date', 'home_team', 'away_team',
                'home_score', 'away_score', 'result']].copy()
home_view.columns = ['date', 'team', 'opponent',
                     'goles_anotados', 'goles_recibidos', 'result_raw']
home_view['gano'] = (home_view['result_raw'] == 'home_win').astype(int)

# Perspectiva del equipo visitante
away_view = df[['date', 'away_team', 'home_team',
                'away_score', 'home_score', 'result']].copy()
away_view.columns = ['date', 'team', 'opponent',
                     'goles_anotados', 'goles_recibidos', 'result_raw']
away_view['gano'] = (away_view['result_raw'] == 'away_win').astype(int)

# Unificamos y ordenamos cronológicamente por equipo
team_history = (
    pd.concat([home_view, away_view])
    .sort_values(['team', 'date'])
    .reset_index(drop=True)
)

print(f"Filas en team_history: {len(team_history):,}")
print(f"(Son el doble de los partidos: {len(df):,} × 2 = {len(df)*2:,})")
print()
print("Ejemplo — últimos partidos de Brasil:")
print(
    team_history[team_history['team'] == 'Brazil']
    .tail(6)[['date', 'team', 'opponent', 
              'goles_anotados', 'goles_recibidos', 'gano']]
    .to_string(index=False)
)

Filas en team_history: 98,954
(Son el doble de los partidos: 49,477 × 2 = 98,954)

Ejemplo — últimos partidos de Brasil:
      date   team opponent  goles_anotados  goles_recibidos  gano
2026-03-31 Brazil  Croatia               3                1     1
2026-05-31 Brazil   Panama               6                2     1
2026-06-06 Brazil    Egypt               2                1     1
2026-06-13 Brazil  Morocco               1                1     0
2026-06-19 Brazil    Haiti               3                0     1
2026-06-24 Brazil Scotland               3                0     1


## 3. Features de forma reciente (rolling)

Calculamos el promedio de los últimos 5 partidos de cada equipo 
usando una ventana deslizante. El `shift(1)` es la protección 
anti-leakage: desplaza los datos un paso, garantizando que el 
resultado de cada partido nunca se incluya en su propio cálculo.

In [4]:
# Calculamos forma reciente por equipo usando ventana de 5 partidos
# shift(1) es clave: desplaza 1 posición hacia adelante para que
# el partido actual NO se incluya en su propio cálculo (anti-leakage)

team_history = team_history.sort_values(['team', 'date']).copy()

team_history['form_5'] = (
    team_history.groupby('team')['gano']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

team_history['goals_scored_avg5'] = (
    team_history.groupby('team')['goles_anotados']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

team_history['goals_conceded_avg5'] = (
    team_history.groupby('team')['goles_recibidos']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

print("✅ Features rolling calculadas")
print()
print("Verificación anti-leakage — historial de Brasil:")
print(
    team_history[team_history['team'] == 'Brazil']
    .tail(8)[['date', 'opponent', 'gano', 
              'form_5', 'goals_scored_avg5', 'goals_conceded_avg5']]
    .to_string(index=False)
)

✅ Features rolling calculadas

Verificación anti-leakage — historial de Brasil:
      date opponent  gano  form_5  goals_scored_avg5  goals_conceded_avg5
2025-11-18  Tunisia     0     0.6                2.4                  0.8
2026-03-26   France     0     0.4                2.0                  1.0
2026-03-31  Croatia     1     0.4                2.2                  1.2
2026-05-31   Panama     1     0.4                1.8                  1.4
2026-06-06    Egypt     1     0.6                2.6                  1.2
2026-06-13  Morocco     0     0.6                2.6                  1.4
2026-06-19    Haiti     1     0.6                2.6                  1.4
2026-06-24 Scotland     1     0.8                3.0                  1.0


In [5]:
# Renombramos para hacer dos joins: uno para local, otro para visitante
home_features = team_history[['date', 'team', 'form_5', 
                               'goals_scored_avg5', 
                               'goals_conceded_avg5']].copy()

home_features.columns = ['date', 'home_team', 
                          'home_form_5',
                          'home_goals_scored_avg5', 
                          'home_goals_conceded_avg5']

away_features = team_history[['date', 'team', 'form_5',
                               'goals_scored_avg5',
                               'goals_conceded_avg5']].copy()

away_features.columns = ['date', 'away_team',
                          'away_form_5',
                          'away_goals_scored_avg5',
                          'away_goals_conceded_avg5']

# Join con el dataset principal
df_features = df_features.merge(home_features, 
                                 on=['date', 'home_team'], 
                                 how='left')
df_features = df_features.merge(away_features, 
                                 on=['date', 'away_team'], 
                                 how='left')

print(f"✅ Features de forma unidas al dataset")
print(f"   Shape: {df_features.shape}")
print()
print("Muestra con features:")
cols = ['date', 'home_team', 'away_team', 
        'home_form_5', 'away_form_5',
        'home_goals_scored_avg5', 'away_goals_scored_avg5']
print(df_features[cols].tail(6).to_string(index=False))

✅ Features de forma unidas al dataset
   Shape: (49819, 20)

Muestra con features:
      date home_team  away_team  home_form_5  away_form_5  home_goals_scored_avg5  away_goals_scored_avg5
2026-06-27  DR Congo Uzbekistan          0.2          0.0                     0.6                     0.4
2026-06-27  Colombia   Portugal          0.8          0.8                     2.0                     2.4
2026-06-27    Panama    England          0.2          0.6                     1.4                     1.6
2026-06-27   Algeria    Austria          0.6          0.8                     1.4                     2.0
2026-06-27    Jordan  Argentina          0.0          1.0                     1.0                     3.0
2026-06-27   Croatia      Ghana          0.4          0.2                     1.2                     0.6


## 4. Unión con el dataset principal

Al unir `team_history` con el dataset original encontramos 342 
filas extra: algunos equipos jugaron 2 partidos el mismo día 
(torneos históricos del siglo XIX). La solución es usar 3 claves 
en el merge en lugar de 2, incluyendo al rival para eliminar 
la ambigüedad.

In [6]:
# Investigamos las duplicadas
print(f"Filas esperadas:  49,477")
print(f"Filas obtenidas:  {len(df_features):,}")
print(f"Diferencia:       {len(df_features) - 49477:,}")
print()

# ¿Hay partidos con misma fecha y mismo par de equipos?
duplicados = df_features[
    df_features.duplicated(subset=['date', 'home_team', 'away_team'], keep=False)
]
print(f"Filas duplicadas por date+home_team+away_team: {len(duplicados):,}")
print()
print("Ejemplos:")
print(
    duplicados[['date', 'home_team', 'away_team', 'tournament']]
    .head(10)
    .to_string(index=False)
)

Filas esperadas:  49,477
Filas obtenidas:  49,819
Diferencia:       342

Filas duplicadas por date+home_team+away_team: 579

Ejemplos:
      date        home_team        away_team                tournament
1890-03-15 Northern Ireland          England British Home Championship
1890-03-15 Northern Ireland          England British Home Championship
1890-03-15            Wales          England British Home Championship
1890-03-15            Wales          England British Home Championship
1891-03-07          England            Wales British Home Championship
1891-03-07          England            Wales British Home Championship
1891-03-07          England Northern Ireland British Home Championship
1891-03-07          England Northern Ireland British Home Championship
1892-03-05 Northern Ireland          England British Home Championship
1892-03-05 Northern Ireland          England British Home Championship


In [7]:
# ¿El dataset original tiene duplicados?
dupes_original = df[
    df.duplicated(subset=['date', 'home_team', 'away_team'], keep=False)
]
print(f"Duplicados en df original: {len(dupes_original)}")
print()

# ¿Y en team_history? (mismo equipo, misma fecha)
dupes_history = team_history[
    team_history.duplicated(subset=['date', 'team'], keep=False)
]
print(f"Equipos con 2 partidos el mismo día en team_history: {len(dupes_history)}")
print()
print("Ejemplos:")
print(
    dupes_history[['date', 'team', 'opponent', 'gano']]
    .head(8)
    .to_string(index=False)
)

Duplicados en df original: 4

Equipos con 2 partidos el mismo día en team_history: 286

Ejemplos:
      date      team opponent  gano
1916-08-15 Argentina  Uruguay     1
1916-08-15 Argentina  Uruguay     1
1916-10-01 Argentina  Uruguay     1
1916-10-01 Argentina  Uruguay     1
1922-10-22 Argentina    Chile     1
1922-10-22 Argentina   Brazil     0
1923-12-02 Argentina   Brazil     0
1923-12-02 Argentina  Uruguay     0


In [8]:
# FIX: reconstruir todo desde cero con merge en 3 claves
# Causa del problema: merge en 2 claves (date + team) genera
# producto cartesiano cuando un equipo jugó 2 partidos el mismo día

# Paso 1: dataset sin los 4 duplicados reales
df_sin_dupes = df.drop_duplicates(
    subset=['date', 'home_team', 'away_team']
).copy()
df_sin_dupes['is_neutral'] = df_sin_dupes['neutral'].astype(int)
df_sin_dupes['is_friendly'] = (df_sin_dupes['tournament'] == 'Friendly').astype(int)

# Paso 2: limpiar team_history de duplicados exactos
team_history_clean = team_history.drop_duplicates(
    subset=['date', 'team', 'opponent']
).copy()

# Paso 3: home/away features CON columna opponent para el merge en 3 claves
home_features = team_history_clean[
    ['date', 'team', 'opponent', 
     'form_5', 'goals_scored_avg5', 'goals_conceded_avg5']
].copy()
home_features.columns = [
    'date', 'home_team', 'away_team',
    'home_form_5', 'home_goals_scored_avg5', 'home_goals_conceded_avg5'
]

away_features = team_history_clean[
    ['date', 'team', 'opponent',
     'form_5', 'goals_scored_avg5', 'goals_conceded_avg5']
].copy()
away_features.columns = [
    'date', 'away_team', 'home_team',
    'away_form_5', 'away_goals_scored_avg5', 'away_goals_conceded_avg5'
]

# Paso 4: merge en 3 claves — elimina la ambigüedad completamente
df_features = df_sin_dupes.merge(
    home_features, on=['date', 'home_team', 'away_team'], how='left'
)
df_features = df_features.merge(
    away_features, on=['date', 'away_team', 'home_team'], how='left'
)

# Verificación
dupes_restantes = df_features.duplicated(
    subset=['date', 'home_team', 'away_team']
).sum()

print(f"Filas originales:       49,477")
print(f"Duplicados eliminados:  4")
print(f"Filas esperadas:        49,473")
print(f"Filas obtenidas:        {len(df_features):,}")
print(f"Duplicados restantes:   {dupes_restantes}")

Filas originales:       49,477
Duplicados eliminados:  4
Filas esperadas:        49,473
Filas obtenidas:        49,475
Duplicados restantes:   0


## 5. Head-to-head (historial entre estos dos equipos)

Para cada partido buscamos cuántas veces se enfrentaron estos 
dos equipos antes de esa fecha, y cuántas veces ganó el local. 
Es la operación más costosa del notebook — itera sobre 49,475 
partidos consultando el historial completo de cada par.

In [9]:
# HEAD-TO-HEAD: historial entre estos dos equipos específicos
# Para cada partido, contamos cuántas veces se enfrentaron antes
# y cuántas ganó cada uno

h2h_list = []

for idx, row in df_features.iterrows():
    fecha        = row['date']
    home         = row['home_team']
    away         = row['away_team']

    # Solo partidos ANTERIORES a este entre estos dos equipos
    historial = df[
        (df['date'] < fecha) &
        (
            ((df['home_team'] == home) & (df['away_team'] == away)) |
            ((df['home_team'] == away) & (df['away_team'] == home))
        )
    ]

    total = len(historial)

    if total == 0:
        h2h_list.append({'h2h_total': 0, 'h2h_home_winrate': 0.5})
        continue

    # Victorias del equipo "home" en este partido, sin importar si fue local
    home_wins = (
        ((historial['home_team'] == home) & (historial['result'] == 'home_win')) |
        ((historial['away_team'] == home) & (historial['result'] == 'away_win'))
    ).sum()

    h2h_list.append({
        'h2h_total':        total,
        'h2h_home_winrate': home_wins / total
    })

h2h_df = pd.DataFrame(h2h_list)
df_features = pd.concat(
    [df_features.reset_index(drop=True), h2h_df], axis=1
)

print(f"✅ Features h2h agregadas")
print()
print("Muestra — partidos con historial rico:")
cols = ['home_team', 'away_team', 'date', 
        'h2h_total', 'h2h_home_winrate']
print(
    df_features[df_features['h2h_total'] > 50]
    [cols].tail(6).to_string(index=False)
)

✅ Features h2h agregadas

Muestra — partidos con historial rico:
  home_team        away_team       date  h2h_total  h2h_home_winrate
     Uganda         Tanzania 2025-12-27         70          0.542857
Switzerland          Germany 2026-03-27         54          0.166667
      Wales Northern Ireland 2026-03-31         94          0.478723
     Norway           Sweden 2026-06-01        109          0.238532
  Lithuania           Latvia 2026-06-06         61          0.311475
    Estonia        Lithuania 2026-06-09         62          0.370968


In [10]:
# Verificación final del dataset de features
print("=== DATASET FINAL DE FEATURES ===")
print(f"Shape: {df_features.shape}")
print()
print("Columnas y nulos:")
print(df_features.isnull().sum()[df_features.isnull().sum() > 0])
print()

# Guardamos
df_features.to_csv('../data/processed/features.csv', index=False)
print("✅ Guardado: data/processed/features.csv")

=== DATASET FINAL DE FEATURES ===
Shape: (49475, 22)

Columnas y nulos:
home_form_5                 141
home_goals_scored_avg5      141
home_goals_conceded_avg5    141
away_form_5                 195
away_goals_scored_avg5      195
away_goals_conceded_avg5    195
dtype: int64

✅ Guardado: data/processed/features.csv


## 6. Tratamiento de nulos

Los primeros partidos de cada selección no tienen historial previo,
por eso generan NaN en las features rolling. Se rellenan con la 
media global de cada columna — el valor neutro más razonable 
cuando no existe información histórica.

In [11]:
# Rellenamos nulos con la media global de cada columna
# Es el valor más neutro cuando no hay historial disponible
cols_rolling = [
    'home_form_5', 'home_goals_scored_avg5', 'home_goals_conceded_avg5',
    'away_form_5', 'away_goals_scored_avg5', 'away_goals_conceded_avg5'
]

for col in cols_rolling:
    media = df_features[col].mean()
    df_features[col] = df_features[col].fillna(media)

# Verificación
nulos_restantes = df_features.isnull().sum().sum()
print(f"Nulos restantes: {nulos_restantes}")
print()
print("Features finales disponibles para el modelo:")
features_modelo = [
    'is_neutral', 'is_friendly',
    'home_form_5', 'home_goals_scored_avg5', 'home_goals_conceded_avg5',
    'away_form_5', 'away_goals_scored_avg5', 'away_goals_conceded_avg5',
    'h2h_total', 'h2h_home_winrate'
]
print(df_features[features_modelo].describe().round(3))

# Guardamos el dataset final limpio
df_features.to_csv('../data/processed/features.csv', index=False)
print()
print("✅ Dataset final guardado sin nulos")

Nulos restantes: 0

Features finales disponibles para el modelo:
       is_neutral  is_friendly  home_form_5  home_goals_scored_avg5  \
count   49475.000    49475.000    49475.000               49475.000   
mean        0.265        0.372        0.391                   1.486   
std         0.441        0.483        0.253                   0.897   
min         0.000        0.000        0.000                   0.000   
25%         0.000        0.000        0.200                   0.800   
50%         0.000        0.000        0.400                   1.400   
75%         1.000        1.000        0.600                   2.000   
max         1.000        1.000        1.000                  17.000   

       home_goals_conceded_avg5  away_form_5  away_goals_scored_avg5  \
count                 49475.000    49475.000               49475.000   
mean                      1.454        0.382                   1.457   
std                       1.004        0.252                   0.885   
min    

## Resumen de features construidas

| Feature | Descripción |
|---|---|
| `is_neutral` | 1 si la cancha es neutral |
| `is_friendly` | 1 si es amistoso |
| `home_form_5` | Tasa de victoria local en últimos 5 partidos |
| `away_form_5` | Tasa de victoria visitante en últimos 5 partidos |
| `home_goals_scored_avg5` | Promedio de goles anotados por local (últimos 5) |
| `home_goals_conceded_avg5` | Promedio de goles recibidos por local (últimos 5) |
| `away_goals_scored_avg5` | Promedio de goles anotados por visitante (últimos 5) |
| `away_goals_conceded_avg5` | Promedio de goles recibidos por visitante (últimos 5) |
| `h2h_total` | Partidos jugados entre estos dos equipos antes de este |
| `h2h_home_winrate` | Tasa histórica de victoria del local contra este rival |

**Regla anti-leakage aplicada:** todas las features rolling usan 
`shift(1)` — el resultado de cada partido solo está disponible 
para el cálculo del partido *siguiente*. 

**Nulos:** los primeros partidos de cada selección no tienen 
historial previo. Se rellenan con la media global, el valor 
neutro más razonable.

**Dataset guardado:** `data/processed/features.csv` — 
49,475 partidos × 22 columnas, listo para el modelo.